# Part 3: Developing an Agentic AI Travel Assistant

In [1]:
# Import Libraries
import requests
from datetime import datetime, timedelta


In [2]:
# API Keys

# Amadeus Flight API
AMADEUS_FLIGHT_API_KEY = "sNXjNQxWU9GjojUc0IxKEvlGrEErUAc5"
AMADEUS_FLIGHT_API_SECRET = "J5vWSR0J5UHOqMWE"
AMADEUS_AIRPORT_API_KEY = "yqGyZjBsnfc97y1DbcAbfnQgSMs4hAov"
AMADEUS_AIRPORT_API_SECRET = "iAl4rGqWJY6L8svM"

# Weather API - WeatherAPI.com
WEATHERAPI_KEY = "f342c1106f304940bcb30926250305"

# Booking.com Rapid API
RAPIDAPI_KEY = "8d200b23d6msh649b99ca34ee4f7p191e6bjsnb498c86a0ba3"


## Flight API Agent

In [3]:
# Flight API Agent
def get_amadeus_token(client_id, client_secret):
    url = "https://test.api.amadeus.com/v1/security/oauth2/token"
    payload = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret
    }
    headers = {'Content-Type': 'application/x-www-form-urlencoded'}
    response = requests.post(url, data=payload, headers=headers).json()
    if 'access_token' not in response:
        print(f"[DEBUG] Failed to get Amadeus token: {response.get('error_description', 'Unknown error')}")
        return None
    return response['access_token']


In [4]:
def get_iata_code(city_name):
    token = get_amadeus_token(AMADEUS_FLIGHT_API_KEY, AMADEUS_FLIGHT_API_SECRET)
    if not token:
        return None
    url = "https://test.api.amadeus.com/v1/reference-data/locations"
    headers = {"Authorization": f"Bearer {token}"}
    params = {
        "keyword": city_name,
        "subType": "CITY"
    }
    response = requests.get(url, headers=headers, params=params).json()
    locations = response.get("data", [])
    if not locations:
        print(f"[DEBUG] No IATA code found for city: {city_name}")
        return None
    return locations[0]["iataCode"]


In [5]:
def get_flights(origin, destination, departure_date):
    if not origin or not destination:
        print(f"[DEBUG] Invalid IATA codes: origin={origin}, destination={destination}")
        return []
    token = get_amadeus_token(AMADEUS_FLIGHT_API_KEY, AMADEUS_FLIGHT_API_SECRET)
    if not token:
        return []
    search_url = "https://test.api.amadeus.com/v2/shopping/flight-offers"
    params = {
        "originLocationCode": origin,
        "destinationLocationCode": destination,
        "departureDate": departure_date,
        "adults": 1,
        "max": 3
    }
    headers = {"Authorization": f"Bearer {token}"}
    response = requests.get(search_url, headers=headers, params=params).json()
    if "data" not in response:
        print(f"[DEBUG] Flight search failed: {response.get('errors', 'Unknown error')}")
        return []
    flights = []
    for offer in response.get("data", []):
        itinerary = offer["itineraries"][0]["segments"][0]
        flights.append({
            "airline": itinerary["carrierCode"],
            "departure": itinerary["departure"]["at"],
            "arrival": itinerary["arrival"]["at"],
            "price": offer["price"]["total"]
        })
    return flights


## Weather API Agent

In [6]:
# Weather API Agent
def get_weather(city, date):
    """
    Fetch weather data from WeatherAPI.com which has a reliable free tier
    that allows forecast searches.
    """
    try:
        url = f"https://api.weatherapi.com/v1/forecast.json?key={WEATHERAPI_KEY}&q={city}&dt={date}&aqi=no"
        
        response = requests.get(url)
        weather_data = response.json()
        
        if response.status_code != 200:
            error_msg = weather_data.get('error', {}).get('message', 'Unknown error')
            print(f"[DEBUG] Weather API error: {error_msg}, Status code: {response.status_code}")
            return {"temp": "N/A", "condition": "No data available", "summary": "Weather data unavailable"}
        
        forecast_day = None
        for day in weather_data.get('forecast', {}).get('forecastday', []):
            if day['date'] == date:
                forecast_day = day
                break
        
        if forecast_day:
            condition = forecast_day['day']['condition']['text']
            avg_temp = forecast_day['day']['avgtemp_c']
            max_temp = forecast_day['day']['maxtemp_c']
            min_temp = forecast_day['day']['mintemp_c']
            
            return {
                "temp": avg_temp,
                "condition": condition,
                "summary": f"{condition} with temps between {min_temp}°C and {max_temp}°C"
            }
        else:
            return {
                "temp": weather_data['current']['temp_c'],
                "condition": weather_data['current']['condition']['text'],
                "summary": f"Current: {weather_data['current']['condition']['text']}"
            }
            
    except Exception as e:
        print(f"[DEBUG] Weather API exception: {str(e)}")
        return {"temp": "N/A", "condition": "Error retrieving data", "summary": "Weather service error"}

## Hotel API Agent

In [7]:
# Hotel API Agent
def get_hotels(city, checkin_date, checkout_date):
    headers = {
        "X-RapidAPI-Key": RAPIDAPI_KEY,
        "X-RapidAPI-Host": "booking-com.p.rapidapi.com"
    }

    # Get destination ID
    location_url = "https://booking-com.p.rapidapi.com/v1/hotels/locations"
    location_params = {"name": city, "locale": "en-us"}
    location_res = requests.get(location_url, headers=headers, params=location_params).json()
    if not location_res or "dest_id" not in location_res[0]:
        print(f"[DEBUG] No dest_id found for {city}")
        return []
    dest_id = location_res[0].get("dest_id")
    dest_type = location_res[0].get("dest_type")
    print(f"[DEBUG] Retrieved dest_id: {dest_id}, dest_type: {dest_type} for {city}")

    # Search for hotels
    hotel_url = "https://booking-com.p.rapidapi.com/v1/hotels/search"
    hotel_params = {
        "checkout_date": checkout_date,
        "checkin_date": checkin_date,
        "dest_id": dest_id,
        "dest_type": dest_type,
        "adults_number": "1",
        "order_by": "popularity",
        "locale": "en-us",
        "units": "metric",
        "room_number": "1",
        "filter_by_currency": "USD",
        "page_number": "0",
        "include_adjacency": "true"
    }
    hotel_res = requests.get(hotel_url, headers=headers, params=hotel_params).json()
    if "message" in hotel_res:
        print(f"[DEBUG] Hotel Search Error: {hotel_res['message']}")
        return []
    results = hotel_res.get("result", [])
    print(f"[DEBUG] Hotels found for {city} ({checkin_date} to {checkout_date}): {len(results)}")
    if not results:
        print(f"[DEBUG] No hotels returned for {city} on {checkin_date}")
        return []

    hotels = []
    for h in results[:5]:
        hotels.append({
            "name": h.get("hotel_name", "Unknown Hotel"),
            "price": f"${h.get('min_total_price', 'N/A')}",
            "availability": "Available" if not h.get("cant_book") else "Not Available",
            "rating": h.get("review_score", "N/A")
        })
    return hotels


## Itenery Planner Agent

In [8]:
# Itinerary Planner Agent
def create_itinerary(origin_city, destination_city, date):
    origin_iata = get_iata_code(origin_city)
    destination_iata = get_iata_code(destination_city)
    flights = get_flights(origin_iata, destination_iata, date)
    weather = get_weather(destination_city, date)
    checkout_date = str(datetime.strptime(date, "%Y-%m-%d").date() + timedelta(days=1))
    hotels = get_hotels(destination_city, date, checkout_date)

    itinerary = {
        "From": origin_city,
        "To": destination_city,
        "Travel Date": date,
        "Weather Forecast": weather,
        "Flight Options": flights,
        "Hotel Options": hotels
    }
    return itinerary


In [9]:
def print_itinerary(itinerary):
    print("\n" + "="*50)
    print("         TRAVEL ITINERARY")
    print("="*50)

    print(f"\nFrom: {itinerary['From']}")
    print(f"To: {itinerary['To']}")
    print(f"Travel Date: {itinerary['Travel Date']}")

    print("\n--- Flight Options ---")
    if not itinerary['Flight Options']:
        print("  No flights found for this route.")
    for i, flight in enumerate(itinerary['Flight Options'], 1):
        print(f"Flight {i}:")
        print(f"  Airline: {flight['airline']}")
        print(f"  Departure: {flight['departure']}")
        print(f"  Arrival: {flight['arrival']}")
        print(f"  Price: ${flight['price']}")

    print("\n--- Weather Forecast ---")
    weather = itinerary['Weather Forecast']
    temp = f"{weather['temp']}°C" if weather['temp'] != "N/A" else "N/A"
    print(f"Temperature: {temp}")
    print(f"Condition: {weather['condition'].capitalize() if weather['condition'] != 'N/A' else 'N/A'}")
    print(f"Summary: {weather['summary']}")

    print("\n--- Hotel Options ---")
    if not itinerary['Hotel Options']:
        print("  No hotels found for this date.")
    else:
        for i, hotel in enumerate(itinerary['Hotel Options'], 1):
            print(f"Hotel {i}:")
            print(f"  Name: {hotel['name']}")
            print(f"  Price: {hotel['price']}")
            print(f"  Availability: {hotel['availability']}")
            print(f"  Rating: {hotel['rating']}")

    print("\n" + "="*50)


In [10]:
# Generate and print itineraries for multiple travel queries
if __name__ == "__main__":
    test_cases = [
        {"origin": "San Francisco", "destination": "New York", "date": "2025-05-03"},
        {"origin": "Los Angeles", "destination": "Miami", "date": "2025-05-04"},
        {"origin": "Chicago", "destination": "London", "date": "2025-05-05"}
    ]
    for test in test_cases:
        print(f"\nTesting itinerary from {test['origin']} to {test['destination']} on {test['date']}")
        full_itinerary = create_itinerary(test["origin"], test["destination"], test["date"])
        print_itinerary(full_itinerary)


Testing itinerary from San Francisco to New York on 2025-05-03
[DEBUG] Retrieved dest_id: 20088325, dest_type: city for New York
[DEBUG] Hotels found for New York (2025-05-03 to 2025-05-04): 20

         TRAVEL ITINERARY

From: San Francisco
To: New York
Travel Date: 2025-05-03

--- Flight Options ---
Flight 1:
  Airline: F9
  Departure: 2025-05-03T00:11:00
  Arrival: 2025-05-03T05:46:00
  Price: $92.96
Flight 2:
  Airline: F9
  Departure: 2025-05-03T21:57:00
  Arrival: 2025-05-04T06:31:00
  Price: $92.96
Flight 3:
  Airline: F9
  Departure: 2025-05-03T13:39:00
  Arrival: 2025-05-03T15:20:00
  Price: $97.89

--- Weather Forecast ---
Temperature: 20.2°C
Condition: Moderate rain
Summary: Moderate rain with temps between 13.4°C and 28.4°C

--- Hotel Options ---
Hotel 1:
  Name: KAMA CENTRAL PARK
  Price: $94.7492
  Availability: Available
  Rating: 7.7
Hotel 2:
  Name: West Side YMCA
  Price: $171.16
  Availability: Available
  Rating: 6.6
Hotel 3:
  Name: Now Now NoHo
  Price: $254.8025

#### NOTE:
Airline Codes:

**F9:** Frontier Airlines.

**FI:** Icelandair.